#  Notebook 01 — Drive Setup & Email Parser Module
### Phishing Email Forensic Analyzer & Link Defanger (Part 1 of 4)

This notebook:
1. Mounts your Google Drive.
2. Creates the shared project folder structure under
   `/content/drive/MyDrive/Phishing_Analyzer_Project/`.
3. Writes the first reusable module, **`email_parser.py`**, into
   `modules/` so every later notebook (and the final Gradio app) can
   simply `import email_parser`.
4. Runs a self-test to prove the module works.

> **Cybersecurity context:** Forensic email analysis always starts with
> reliable *parsing*. Real-world phishing emails are frequently
> multipart MIME messages (plain text + HTML alternative, sometimes with
> attachments), and headers can be missing, spoofed, or malformed. Our
> parser must never crash on messy input — it should degrade gracefully
> and report what it *couldn't* find rather than throwing an exception.

## 1. Mount Google Drive & Build Folder Structure

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = "/content/drive/MyDrive/Phishing_Analyzer_Project"
MODULES_DIR = os.path.join(PROJECT_ROOT, "modules")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

os.makedirs(MODULES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Modules dir :", MODULES_DIR)
print("Data dir    :", DATA_DIR)
print("\nFolder contents:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Phishing_Analyzer_Project
Modules dir : /content/drive/MyDrive/Phishing_Analyzer_Project/modules
Data dir    : /content/drive/MyDrive/Phishing_Analyzer_Project/data

Folder contents:
Phishing_Analyzer_Project/
  modules/
    email_parser.py
    nlp_analyzer.py
    risk_scorer.py
    link_defanger.py
    __pycache__/
      email_parser.cpython-313.pyc
      risk_scorer.cpython-313.pyc
      nlp_analyzer.cpython-313.pyc
      link_defanger.cpython-313.pyc
  data/


## 2. Write `email_parser.py` to Drive

This module accepts **either**:
- a `str` containing raw RFC822 email source (e.g. pasted into a text box), **or**
- `bytes` (e.g. the raw contents of an uploaded `.eml` file)

and returns a clean, predictable dictionary of headers + body text,
regardless of whether the original message was multipart, HTML-only,
plain-text-only, or missing headers entirely.

In [2]:
%%writefile /content/drive/MyDrive/Phishing_Analyzer_Project/modules/email_parser.py

import re
from email import policy
from email.parser import BytesParser, Parser


def parse_email(raw_input):
    """
    Parse a raw email and extract headers + body content.

    Parameters
    ----------
    raw_input : str or bytes
        Raw RFC822 email source. Pass a `str` for pasted email text,
        or `bytes` for the raw contents of an uploaded .eml file.

    Returns
    -------
    dict with keys:
        headers      : dict with from/to/subject/date (each "N/A" if missing)
        body_text    : str  (plain-text body, or HTML-stripped fallback)
        body_html    : str  (raw HTML body if present, else "")
        attachments  : list[str]  (attachment filenames, if any)
        error        : str or None  (non-fatal parsing warning/error)
    """
    result = {
        "headers": {"from": "N/A", "to": "N/A", "subject": "N/A", "date": "N/A"},
        "body_text": "",
        "body_html": "",
        "attachments": [],
        "error": None,
    }

    # --- Guard clause: empty / missing input ---
    if raw_input is None or (isinstance(raw_input, (str, bytes)) and len(raw_input) == 0):
        result["error"] = "Empty email input provided."
        return result

    # --- Parse using the appropriate parser for str vs bytes ---
    try:
        if isinstance(raw_input, bytes):
            msg = BytesParser(policy=policy.default).parsebytes(raw_input)
        elif isinstance(raw_input, str):
            msg = Parser(policy=policy.default).parsestr(raw_input)
        else:
            result["error"] = f"Unsupported input type: {type(raw_input)}"
            return result
    except Exception as e:
        result["error"] = f"Failed to parse email: {e}"
        return result

    # --- Headers (use .get with a default so missing headers never raise) ---
    result["headers"]["from"] = msg.get("From", "N/A") or "N/A"
    result["headers"]["to"] = msg.get("To", "N/A") or "N/A"
    result["headers"]["subject"] = msg.get("Subject", "N/A") or "N/A"
    result["headers"]["date"] = msg.get("Date", "N/A") or "N/A"

    # --- Body extraction: handle multipart (text + html + attachments) ---
    try:
        if msg.is_multipart():
            for part in msg.walk():
                content_type = part.get_content_type()
                content_disposition = str(part.get("Content-Disposition", ""))

                # Attachments: just record the filename, don't extract binary content
                if "attachment" in content_disposition:
                    filename = part.get_filename()
                    if filename:
                        result["attachments"].append(filename)
                    continue

                try:
                    payload = part.get_content()
                except Exception:
                    # Some parts (e.g. images, unusual encodings) may fail to decode as text
                    continue

                if content_type == "text/plain" and not result["body_text"]:
                    result["body_text"] = payload
                elif content_type == "text/html" and not result["body_html"]:
                    result["body_html"] = payload
        else:
            content_type = msg.get_content_type()
            payload = msg.get_content()
            if content_type == "text/html":
                result["body_html"] = payload
            else:
                result["body_text"] = payload
    except Exception as e:
        result["error"] = f"Body extraction warning: {e}"

    # --- Fallback: if only HTML was found, produce a crude plain-text view ---
    # Phishing kits often send HTML-only bodies to hide raw link text from
    # simple text scanners, so we still want *something* to run NLP/link
    # analysis against.
    if not result["body_text"] and result["body_html"]:
        stripped = re.sub("<[^<]+?>", " ", result["body_html"])
        result["body_text"] = re.sub(r"\s+", " ", stripped).strip()

    return result


def get_full_text_for_analysis(parsed_email):
    """Combine subject + body_text into one blob for keyword/link scanning."""
    subject = (parsed_email.get("headers", {}) or {}).get("subject", "") or ""
    body = parsed_email.get("body_text", "") or ""
    return f"{subject}\n{body}"


Overwriting /content/drive/MyDrive/Phishing_Analyzer_Project/modules/email_parser.py


## 3. Self-Test — Import the Module and Parse a Sample Email

In [3]:
import sys
sys.path.append(MODULES_DIR)

import email_parser
import importlib
importlib.reload(email_parser)  # ensures we pick up the freshest version on Drive

SAMPLE_MULTIPART_EMAIL = '''From: "PayPal Security" <service@paypa1-verify.com>
To: victim@example.com
Subject: Urgent: Your account has been suspended
Date: Mon, 1 Jan 2024 09:00:00 -0000
Content-Type: multipart/alternative; boundary="BOUNDARY"

--BOUNDARY
Content-Type: text/plain; charset="utf-8"

Dear Customer,

Your account has been suspended due to suspicious activity.
You must verify your account within 24 hours or it will be permanently closed.

Click here to verify: http://paypa1-verify.com/login

--BOUNDARY
Content-Type: text/html; charset="utf-8"

<html><body><p>Dear Customer,</p>
<p>Your account has been <b>suspended</b>. Click
<a href="http://paypa1-verify.com/login">here</a> to verify.</p>
</body></html>
--BOUNDARY--
'''

parsed = email_parser.parse_email(SAMPLE_MULTIPART_EMAIL)
print("Headers   :", parsed["headers"])
print("Error     :", parsed["error"])
print("Body text :", parsed["body_text"][:200])
print("\nFull text for downstream analysis:")
print(email_parser.get_full_text_for_analysis(parsed))

print("\n--- Edge case: empty input ---")
print(email_parser.parse_email(""))

Headers   : {'from': 'PayPal Security <service@paypa1-verify.com>', 'to': 'victim@example.com', 'subject': 'Urgent: Your account has been suspended', 'date': 'Mon, 01 Jan 2024 09:00:00 -0000'}
Error     : None
Body text : Dear Customer,

Your account has been suspended due to suspicious activity.
You must verify your account within 24 hours or it will be permanently closed.

Click here to verify: http://paypa1-verify.c

Full text for downstream analysis:
Urgent: Your account has been suspended
Dear Customer,

Your account has been suspended due to suspicious activity.
You must verify your account within 24 hours or it will be permanently closed.

Click here to verify: http://paypa1-verify.com/login


--- Edge case: empty input ---
{'headers': {'from': 'N/A', 'to': 'N/A', 'subject': 'N/A', 'date': 'N/A'}, 'body_text': '', 'body_html': '', 'attachments': [], 'error': 'Empty email input provided.'}
